# AMR Drug Repurposing — 04: Scientific Fixes

Addresses five critical issues identified in the analysis:

1. **Antibiotic exclusion filter broken** — `indication_class` NaN for most drugs; fix uses ATC J01 codes + drug name keyword filter
2. **Data leakage in train/test split** — random row split allows same molecule across train/test; fix splits by unique `molecule_chembl_id`
3. **SHAP plot broken** — interaction values returned instead of marginal SHAP; fix forces correct output shape
4. **Duplicate salt forms** — same parent compound counted multiple times; fix deduplicates by InChIKey first block
5. **MLP underperforms RF** — rerun with corrected data; reframe contribution honestly


## Paths & imports

In [3]:
from pathlib import Path
import os, ast, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

from rdkit import Chem
from rdkit.Chem import AllChem, Draw
from rdkit.Chem.inchi import MolToInchi
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              classification_report, confusion_matrix,
                              roc_curve, precision_recall_curve)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torch.optim as optim

PROJECT_ROOT = Path.cwd().parent if (Path.cwd().parent / 'data').exists() else Path.cwd()
DATA_DIR  = PROJECT_ROOT / 'data'
CKPT_DIR  = PROJECT_ROOT / 'checkpoints'
FIG_DIR   = PROJECT_ROOT / 'figures'

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('mps' if torch.backends.mps.is_available()
                       else 'cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"Project root: {PROJECT_ROOT}")

Device: mps
Project root: /Users/jubayer/Projects/amr_drug_repurposing


## Fix 1 — Molecule-level train/test split

**Problem:** Random row-level split allows the same molecule (tested against multiple organisms)
to appear in both train and test sets, inflating all metrics.

**Fix:** Split on unique `molecule_chembl_id`. All rows for a given molecule go to either
train or test — never both.


In [4]:
df_clean = pd.read_csv(DATA_DIR / 'eskape_clean.csv', low_memory=False)
X_all    = np.load(DATA_DIR / 'X_ecfp.npy')
y_all    = np.load(DATA_DIR / 'y_labels.npy')

print(f"Total rows      : {len(df_clean):,}")
print(f"Unique molecules: {df_clean['molecule_chembl_id'].nunique():,}")
print(f"Avg rows/mol    : {len(df_clean)/df_clean['molecule_chembl_id'].nunique():.2f}")

# Split unique molecule IDs — stratify by majority label per molecule
mol_label = (df_clean.groupby('molecule_chembl_id')['active']
               .agg(lambda x: int(x.mean() >= 0.5)))  # majority vote label

mol_ids = mol_label.index.to_numpy().astype(str)  # force plain numpy, not PyArrow-backed
mol_y   = mol_label.to_numpy().astype(int)

train_mols, test_mols = train_test_split(
    mol_ids, test_size=0.2, random_state=SEED, stratify=mol_y
)

train_mask = df_clean['molecule_chembl_id'].isin(train_mols).values
test_mask  = df_clean['molecule_chembl_id'].isin(test_mols).values

X_train = X_all[train_mask]
y_train = y_all[train_mask]
X_test  = X_all[test_mask]
y_test  = y_all[test_mask]

# Verify zero overlap
train_set = set(df_clean.loc[train_mask, 'molecule_chembl_id'])
test_set  = set(df_clean.loc[test_mask,  'molecule_chembl_id'])
overlap   = train_set & test_set
print(f"\nTrain rows : {X_train.shape[0]:,} | Test rows : {X_test.shape[0]:,}")
print(f"Train mols : {len(train_mols):,}  | Test mols : {len(test_mols):,}")
print(f"Molecule overlap between train/test : {len(overlap)} (must be 0)")
print(f"Train positive rate : {y_train.mean():.3f}")
print(f"Test  positive rate : {y_test.mean():.3f}")

Task was destroyed but it is pending!
task: <Task pending name='Task-68' coro=<_async_in_context.<locals>.run_in_context() done, defined at /Users/jubayer/Projects/amr_drug_repurposing/.venv/lib/python3.11/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-69' coro=<Kernel.shell_main() running at /Users/jubayer/Projects/amr_drug_repurposing/.venv/lib/python3.11/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /Users/jubayer/Projects/amr_drug_repurposing/.venv/lib/python3.11/site-packages/zmq/eventloop/zmqstream.py:563]>
Task was destroyed but it is pending!
task: <Task pending name='Task-69' coro=<Kernel.shell_main() running at /Users/jubayer/Projects/amr_drug_repurposing/.venv/lib/python3.11/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]>


Total rows      : 71,623
Unique molecules: 48,153
Avg rows/mol    : 1.49

Train rows : 57,572 | Test rows : 14,051
Train mols : 38,522  | Test mols : 9,631
Molecule overlap between train/test : 0 (must be 0)
Train positive rate : 0.431
Test  positive rate : 0.431


## Fix 2 — Correct antibiotic exclusion using ATC codes + name keywords

**Problem:** `indication_class` column is NaN for almost all drugs in ChEMBL.
The original filter passed known antibiotics (fluoroquinolones, carbapenems, etc.)
directly into the repurposing screen.

**Fix:**
- ATC J01 prefix = systemic antibacterials (WHO standard)
- ATC J04 prefix = antimycobacterials
- Drug name keyword list as fallback for drugs without ATC codes
- Deduplicate salt forms by InChIKey parent (first block)


In [8]:
df_drugs = pd.read_csv(DATA_DIR / 'approved_drugs.csv', low_memory=False)
print(f"Approved drugs loaded: {len(df_drugs):,}")

# ── Parse ATC codes from list-string column
def parse_atc(val):
    if pd.isna(val):
        return []
    try:
        return ast.literal_eval(val) if isinstance(val, str) else list(val)
    except:
        return []

df_drugs['atc_list'] = df_drugs['atc_classifications'].apply(parse_atc)

# ── Flag antibiotics by ATC J01/J04
ANTIBIOTIC_ATC_PREFIXES = ('J01', 'J04')

def is_antibiotic_atc(atc_list):
    return any(code.startswith(ANTIBIOTIC_ATC_PREFIXES) for code in atc_list)

df_drugs['is_abx_atc'] = df_drugs['atc_list'].apply(is_antibiotic_atc)

# ── Flag antibiotics by drug name keywords (fallback)
ABX_NAME_KW = [
    'floxacin', 'oxacin', 'cycline', 'mycin', 'cillin', 'penem',
    'cef', 'ceph', 'tazobactam', 'sulbactam', 'avibactam', 'relebactam',
    'fosfomycin', 'rifamp', 'rifabutin', 'linezolid', 'tedizolid',
    'vancomycin', 'teicoplanin', 'dalbavancin', 'oritavancin',
    'colistin', 'polymyxin', 'tigecycline', 'chloramphenicol',
    'trimethoprim', 'sulfameth', 'nitrofurantoin', 'metronidazole',
    'fidaxomicin', 'retapamulin', 'lefamulin', 'mupirocin',
    'nalidixic', 'novobiocin', 'pretomanid', 'delamanid', 'bedaquiline',
    'plazomicin', 'netilmicin', 'amikacin', 'gentamicin', 'tobramycin',
    'neomycin', 'spectinomycin', 'streptomycin', 'kanamycin',
]

def is_antibiotic_name(name):
    if not isinstance(name, str):
        return False
    n = name.lower()
    return any(kw in n for kw in ABX_NAME_KW)

df_drugs['is_abx_name'] = df_drugs['pref_name'].apply(is_antibiotic_name)
df_drugs['is_antibiotic'] = df_drugs['is_abx_atc'] | df_drugs['is_abx_name']

print(f"Flagged as antibiotic (ATC J01/J04) : {df_drugs['is_abx_atc'].sum()}")
print(f"Flagged as antibiotic (name keyword): {df_drugs['is_abx_name'].sum()}")
print(f"Total flagged as antibiotic         : {df_drugs['is_antibiotic'].sum()}")
print(f"Non-antibiotic drugs remaining      : {(~df_drugs['is_antibiotic']).sum()}")

Approved drugs loaded: 3,280
Flagged as antibiotic (ATC J01/J04) : 147
Flagged as antibiotic (name keyword): 260
Total flagged as antibiotic         : 287
Non-antibiotic drugs remaining      : 2993


## Fix 3 — Deduplicate salt forms by InChIKey parent

**Problem:** LEVOFLOXACIN, LEVOFLOXACIN ANHYDROUS, LEVOFLOXACIN HYDROCHLORIDE
are three separate rows — all predicted with near-identical scores, inflating hit counts.

**Fix:** Generate InChIKey for each SMILES, deduplicate on the first block
(connectivity layer, salt-invariant), keep highest ensemble score per parent.


In [9]:
def smiles_to_ecfp(smiles, radius=2, nbits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=nbits)
    return np.array(fp)

def smiles_to_inchikey_parent(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    inchi = MolToInchi(mol)
    if inchi is None:
        return None
    # InChIKey first block = connectivity layer (salt-invariant)
    key = Chem.InchiInfo.InchiToInchiKey(inchi) if hasattr(Chem, 'InchiInfo') else None
    if key is None:
        try:
            from rdkit.Chem.inchi import InchiToInchiKey
            key = InchiToInchiKey(inchi)
        except:
            return None
    return key.split('-')[0] if key else None

df_nonabx = df_drugs[~df_drugs['is_antibiotic']].copy()
df_nonabx = df_nonabx.dropna(subset=['canonical_smiles'])
df_nonabx = df_nonabx[df_nonabx['canonical_smiles'].str.len() > 5]

print(f"Non-antibiotic drugs with SMILES: {len(df_nonabx):,}")
print("Computing InChIKey parents (deduplication)...")

df_nonabx['inchikey_parent'] = df_nonabx['canonical_smiles'].apply(smiles_to_inchikey_parent)
df_nonabx = df_nonabx.dropna(subset=['inchikey_parent'])

# Keep one representative SMILES per parent (prefer shorter/cleaner name)
df_nonabx_dedup = (df_nonabx.sort_values('pref_name')
                             .drop_duplicates(subset='inchikey_parent', keep='first')
                             .reset_index(drop=True))

print(f"Before dedup: {len(df_nonabx):,}")
print(f"After dedup : {len(df_nonabx_dedup):,} unique parent structures")
print(f"Removed     : {len(df_nonabx)-len(df_nonabx_dedup):,} salt/solvate duplicates")

Non-antibiotic drugs with SMILES: 2,827
Computing InChIKey parents (deduplication)...
Before dedup: 2,827
After dedup : 2,734 unique parent structures
Removed     : 93 salt/solvate duplicates


## Retrain Random Forest on corrected (molecule-level) split

In [10]:
print("Training Random Forest on molecule-level split...")
t0 = time.time()

rf = RandomForestClassifier(
    n_estimators=500, max_depth=None, min_samples_leaf=2,
    class_weight='balanced', n_jobs=-1, random_state=SEED
)
rf.fit(X_train, y_train)

rf_prob = rf.predict_proba(X_test)[:, 1]
rf_pred = (rf_prob >= 0.5).astype(int)
rf_roc  = roc_auc_score(y_test, rf_prob)
rf_prc  = average_precision_score(y_test, rf_prob)

print(f"RF ({time.time()-t0:.0f}s)  ROC-AUC: {rf_roc:.4f}  PRC-AUC: {rf_prc:.4f}")
print(classification_report(y_test, rf_pred, target_names=['Inactive','Active']))

Training Random Forest on molecule-level split...
RF (17s)  ROC-AUC: 0.9194  PRC-AUC: 0.9016
              precision    recall  f1-score   support

    Inactive       0.85      0.87      0.86      7990
      Active       0.82      0.80      0.81      6061

    accuracy                           0.84     14051
   macro avg       0.84      0.83      0.84     14051
weighted avg       0.84      0.84      0.84     14051



## Retrain MLP on corrected split

In [11]:
class FingerprintDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

class AntibacterialMLP(nn.Module):
    def __init__(self, input_dim=2048, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 1024), nn.BatchNorm1d(1024), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(1024, 512),       nn.BatchNorm1d(512),  nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(512, 256),        nn.BatchNorm1d(256),  nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 128),        nn.BatchNorm1d(128),  nn.ReLU(), nn.Dropout(dropout*0.5),
            nn.Linear(128, 1),
        )
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)
    def forward(self, x): return self.net(x).squeeze(-1)

n_neg, n_pos = (y_train==0).sum(), (y_train==1).sum()
pos_weight   = torch.tensor([n_neg/n_pos], dtype=torch.float32).to(device)
sample_wts   = np.where(y_train==1, n_neg/n_pos, 1.0)
sampler      = WeightedRandomSampler(sample_wts, len(y_train), replacement=True)

BATCH = 256
train_loader = DataLoader(FingerprintDataset(X_train, y_train), batch_size=BATCH, sampler=sampler)
test_loader  = DataLoader(FingerprintDataset(X_test,  y_test),  batch_size=BATCH, shuffle=False)

model     = AntibacterialMLP().to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=60, eta_min=1e-5)

def get_proba(model, loader):
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb.to(device)).cpu().numpy()
            probs.extend(1/(1+np.exp(-logits)))
            labels.extend(yb.numpy())
    return np.array(probs), np.array(labels)

CKPT = CKPT_DIR / 'best_mlp_fixed.pt'
EPOCHS, LOG_EVERY = 80, 10
best_auc = 0.0
train_losses, val_aucs, val_prcs = [], [], []

print(f"Training MLP on {device} for {EPOCHS} epochs...")
for epoch in range(EPOCHS):
    model.train()
    ep_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        ep_loss += loss.item()
    scheduler.step()

    avg_loss = ep_loss / len(train_loader)
    train_losses.append(avg_loss)
    p, l = get_proba(model, test_loader)
    roc = roc_auc_score(l, p)
    prc = average_precision_score(l, p)
    val_aucs.append(roc)
    val_prcs.append(prc)

    if roc > best_auc:
        best_auc = roc
        torch.save({'epoch':epoch, 'model_state':model.state_dict(),
                    'best_auc':best_auc}, CKPT)
        marker = " ← best"
    else:
        marker = ""

    if (epoch+1) % LOG_EVERY == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | Loss: {avg_loss:.4f} | "
              f"ROC-AUC: {roc:.4f} | PRC-AUC: {prc:.4f}{marker}")

print(f"\nBest MLP ROC-AUC: {best_auc:.4f}")

Training MLP on mps for 80 epochs...
Epoch  10/80 | Loss: 0.2629 | ROC-AUC: 0.9106 | PRC-AUC: 0.8872
Epoch  20/80 | Loss: 0.2057 | ROC-AUC: 0.9096 | PRC-AUC: 0.8820
Epoch  30/80 | Loss: 0.1803 | ROC-AUC: 0.9098 | PRC-AUC: 0.8822
Epoch  40/80 | Loss: 0.1610 | ROC-AUC: 0.9097 | PRC-AUC: 0.8815
Epoch  50/80 | Loss: 0.1492 | ROC-AUC: 0.9081 | PRC-AUC: 0.8771
Epoch  60/80 | Loss: 0.1477 | ROC-AUC: 0.9082 | PRC-AUC: 0.8783
Epoch  70/80 | Loss: 0.1495 | ROC-AUC: 0.9081 | PRC-AUC: 0.8785
Epoch  80/80 | Loss: 0.1543 | ROC-AUC: 0.9071 | PRC-AUC: 0.8783

Best MLP ROC-AUC: 0.9121


## Load best MLP and compute final test metrics

In [12]:
ckpt = torch.load(CKPT_DIR / 'best_mlp_fixed.pt', map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state'])
mlp_prob, y_true = get_proba(model, test_loader)
mlp_pred = (mlp_prob >= 0.5).astype(int)
mlp_roc  = roc_auc_score(y_true, mlp_prob)
mlp_prc  = average_precision_score(y_true, mlp_prob)

print("=" * 55)
print("  CORRECTED RESULTS (molecule-level split)")
print("=" * 55)
print(f"{'Model':<20} {'ROC-AUC':>10} {'PRC-AUC':>10}")
print(f"{'Random Forest':<20} {rf_roc:>10.4f} {rf_prc:>10.4f}")
print(f"{'Deep MLP':<20} {mlp_roc:>10.4f} {mlp_prc:>10.4f}")
print()
print("MLP Classification Report:")
print(classification_report(y_true, mlp_pred, target_names=['Inactive','Active']))

  CORRECTED RESULTS (molecule-level split)
Model                   ROC-AUC    PRC-AUC
Random Forest            0.9194     0.9016
Deep MLP                 0.9121     0.8909

MLP Classification Report:
              precision    recall  f1-score   support

    Inactive       0.87      0.83      0.85      7990
      Active       0.79      0.83      0.81      6061

    accuracy                           0.83     14051
   macro avg       0.83      0.83      0.83     14051
weighted avg       0.84      0.83      0.83     14051



## Fix 4 — Corrected repurposing screen

Uses deduplicated non-antibiotic drug set (ATC J01/J04 excluded + name keywords).


In [13]:
print("Generating fingerprints for deduplicated non-antibiotic drugs...")
screen_fps, screen_ids, screen_names, screen_smiles, screen_atc = [], [], [], [], []

for _, row in tqdm(df_nonabx_dedup.iterrows(), total=len(df_nonabx_dedup)):
    fp = smiles_to_ecfp(row['canonical_smiles'])
    if fp is not None:
        screen_fps.append(fp)
        screen_ids.append(row['molecule_chembl_id'])
        screen_names.append(row.get('pref_name', 'Unknown'))
        screen_smiles.append(row['canonical_smiles'])
        screen_atc.append(str(row.get('atc_list', [])))

X_screen = np.array(screen_fps, dtype=np.float32)
print(f"Screen matrix: {X_screen.shape}")

# MLP predictions
model.eval()
screen_probs = []
with torch.no_grad():
    for batch in DataLoader(torch.tensor(X_screen), batch_size=512, shuffle=False):
        logits = model(batch.to(device)).cpu().numpy()
        screen_probs.extend(1/(1+np.exp(-logits)))

rf_screen_probs = rf.predict_proba(X_screen)[:, 1]
ensemble        = (np.array(screen_probs) + rf_screen_probs) / 2

df_hits = pd.DataFrame({
    'chembl_id':     screen_ids,
    'drug_name':     screen_names,
    'smiles':        screen_smiles,
    'atc_codes':     screen_atc,
    'prob_mlp':      np.round(screen_probs, 4),
    'prob_rf':       np.round(rf_screen_probs, 4),
    'prob_ensemble': np.round(ensemble, 4),
}).sort_values('prob_ensemble', ascending=False).reset_index(drop=True)

df_hits.to_csv(DATA_DIR / 'repurposing_candidates_fixed.csv', index=False)

print(f"\nDrugs screened  : {len(df_hits):,}")
print(f"p >= 0.9        : {(df_hits.prob_ensemble>=0.9).sum()}")
print(f"p >= 0.8        : {(df_hits.prob_ensemble>=0.8).sum()}")
print(f"p >= 0.7        : {(df_hits.prob_ensemble>=0.7).sum()}")
print(f"\nTop 20 repurposing candidates (true non-antibiotics):")
print(df_hits[['drug_name','atc_codes','prob_ensemble','prob_mlp','prob_rf']]
      .head(20).to_string(index=False))

Generating fingerprints for deduplicated non-antibiotic drugs...


  0%|          | 0/2734 [00:00<?, ?it/s]

Screen matrix: (2734, 2048)

Drugs screened  : 2,734
p >= 0.9        : 3
p >= 0.8        : 12
p >= 0.7        : 35

Top 20 repurposing candidates (true non-antibiotics):
             drug_name                                                                                           atc_codes  prob_ensemble  prob_mlp  prob_rf
   MOXALACTAM DISODIUM                                                                                                  []         0.9219    0.9823   0.8615
          METHOTREXATE                                                                              ['L04AX03', 'L01BA01']         0.9155    0.9996   0.8314
          PRALATREXATE                                                                                         ['L01BA05']         0.9133    0.9972   0.8293
             TRICLOSAN                                                                              ['D08AE04', 'D09AA06']         0.8785    0.9648   0.7921
             ETRASIMOD                       

## Fix 5 — Correct SHAP computation

**Problem:** `shap.TreeExplainer` with a background dataset returns interaction-style
output (3D array) instead of marginal SHAP values (2D).

**Fix:** Use `check_additivity=False` and explicitly extract class-1 marginal values.
Verify shape is `(n_samples, n_features)` before plotting.


In [ ]:
import shap

print("Computing SHAP values...")
# No background = faster Tree SHAP (path-dependent, correct for RF)
explainer = shap.TreeExplainer(rf, feature_perturbation="tree_path_dependent")
X_explain = X_test[:300]  # 300 sufficient for stable mean|SHAP|; 1000 too slow on 500-tree RF

shap_vals = explainer.shap_values(X_explain)

# shap_values returns list [class0, class1] for binary RF
print(f"shap_vals type  : {type(shap_vals)}")
if isinstance(shap_vals, list):
    print(f"len(shap_vals)  : {len(shap_vals)}")
    print(f"shap_vals[1] shape: {np.array(shap_vals[1]).shape}")
    sv = np.array(shap_vals[1])   # class 1 (active)
else:
    sv = np.array(shap_vals)

assert sv.ndim == 2, f"Expected 2D SHAP array, got shape {sv.shape}"
print(f"Final sv shape  : {sv.shape}  (n_samples × n_features) ✓")

# Mean absolute SHAP per feature
mean_abs = np.abs(sv).mean(axis=0)
top_bits  = np.argsort(mean_abs)[::-1][:20]
print(f"\nTop 5 ECFP4 bits by |SHAP|:")
for i, bit in enumerate(top_bits[:5]):
    print(f"  Rank {i+1}: bit_{bit:<5d}  mean|SHAP|={mean_abs[bit]:.5f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
shap.summary_plot(
    sv, X_explain,
    feature_names=[f"bit_{i}" for i in range(X_train.shape[1])],
    max_display=20,
    show=False,
    plot_size=None,
)
ax = plt.gca()
ax.set_title("SHAP Feature Importance — Top 20 ECFP4 bits (RF, active class)", fontsize=10)
ax.set_xlabel("SHAP value (impact on antibacterial activity prediction)")
plt.tight_layout()
plt.savefig(FIG_DIR / 'shap_fixed.png', dpi=300, bbox_inches='tight')
plt.savefig(FIG_DIR / 'shap_fixed.svg', bbox_inches='tight')
plt.show()
print("Saved shap_fixed.png / .svg")

## Summary figure — corrected model performance + repurposing candidates

In [ ]:
plt.rcParams.update({
    'font.family':'sans-serif','font.sans-serif':['Arial','Helvetica','DejaVu Sans'],
    'font.size':9,'axes.titlesize':10,'axes.labelsize':9,
    'xtick.labelsize':8,'ytick.labelsize':8,'legend.fontsize':8,
    'axes.linewidth':0.8,'lines.linewidth':1.5,
    'axes.spines.top':False,'axes.spines.right':False,
    'savefig.dpi':300,'savefig.bbox':'tight',
})

C = {'mlp':'#1B7837','rf':'#762A83','neutral':'#878787','highlight':'#D6604D','active':'#2166AC'}

fig = plt.figure(figsize=(7.2, 5.5))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.4)

ax1 = fig.add_subplot(gs[0, 0])   # ROC
ax2 = fig.add_subplot(gs[0, 1])   # PR
ax3 = fig.add_subplot(gs[0, 2])   # Confusion matrix
ax4 = fig.add_subplot(gs[1, :])   # Top candidates

# ── ROC
fpr_m, tpr_m, _ = roc_curve(y_true, mlp_prob)
fpr_r, tpr_r, _ = roc_curve(y_test,  rf_prob)
ax1.plot(fpr_m, tpr_m, color=C['mlp'], lw=1.5, label=f'MLP (AUC={mlp_roc:.3f})')
ax1.plot(fpr_r, tpr_r, color=C['rf'],  lw=1.5, ls='--', label=f'RF  (AUC={rf_roc:.3f})')
ax1.plot([0,1],[0,1], color=C['neutral'], lw=0.8, ls=':', alpha=0.6)
ax1.fill_between(fpr_m, tpr_m, alpha=0.07, color=C['mlp'])
ax1.set(xlabel='FPR', ylabel='TPR', title='A  ROC curve', xlim=(0,1), ylim=(0,1))
ax1.legend(frameon=False, handlelength=1.2)
ax1.title.set_fontweight('bold')

# ── PR
prec_m, rec_m, _ = precision_recall_curve(y_true, mlp_prob)
prec_r, rec_r, _ = precision_recall_curve(y_test,  rf_prob)
ax2.plot(rec_m, prec_m, color=C['mlp'], lw=1.5, label=f'MLP (AUC={mlp_prc:.3f})')
ax2.plot(rec_r, prec_r, color=C['rf'],  lw=1.5, ls='--', label=f'RF  (AUC={rf_prc:.3f})')
ax2.set(xlabel='Recall', ylabel='Precision', title='B  Precision-recall')
ax2.legend(frameon=False, handlelength=1.2)
ax2.title.set_fontweight('bold')

# ── Confusion matrix (MLP)
cm = confusion_matrix(y_true, mlp_pred)
im = ax3.imshow(cm, cmap='Blues')
ax3.set_xticks([0,1]); ax3.set_yticks([0,1])
ax3.set_xticklabels(['Inact.','Act.']); ax3.set_yticklabels(['Inact.','Act.'])
ax3.set_xlabel('Predicted'); ax3.set_ylabel('True')
ax3.set_title('C  Confusion matrix', fontweight='bold')
for i in range(2):
    for j in range(2):
        ax3.text(j, i, f'{cm[i,j]:,}', ha='center', va='center',
                 color='white' if cm[i,j]>cm.max()*0.5 else 'black', fontsize=8)
fig.colorbar(im, ax=ax3, shrink=0.75)

# ── Top 20 repurposing candidates (horizontal bar)
top20 = df_hits.head(20).copy()
scores = top20['prob_ensemble'].values
colors = [C['mlp'] if s>=0.8 else C['highlight'] if s>=0.6 else C['neutral'] for s in scores]
ax4.barh(range(len(top20)), scores, color=colors, edgecolor='none', height=0.7)
ax4.set_yticks(range(len(top20)))
ax4.set_yticklabels([f"{row['drug_name']}  |  {str(row['atc_codes'])[:25]}"
                      for _, row in top20.iterrows()], fontsize=6.5)
ax4.axvline(0.5, color='red',   ls='--', lw=0.8, label='0.5 threshold')
ax4.axvline(0.8, color=C['mlp'], ls='--', lw=0.8, label='0.8 threshold')
ax4.set_xlabel('Ensemble probability (antibacterial activity)')
ax4.set_title('D  Top 20 repurposing candidates (non-antibiotics, deduplicated)', fontweight='bold')
ax4.invert_yaxis()
ax4.set_xlim(0, 1.05)
ax4.legend(frameon=False, fontsize=7, loc='lower right')

plt.savefig(FIG_DIR / 'fig_corrected_summary.png', dpi=300, bbox_inches='tight')
plt.savefig(FIG_DIR / 'fig_corrected_summary.svg', bbox_inches='tight')
plt.show()
print("Saved fig_corrected_summary.png / .svg")

## Validation — compare original vs corrected metrics

In [ ]:
# Original (row-level split) metrics from notebook 02
orig_rf_roc  = 0.9007   # from assessment
orig_mlp_roc = 0.913

summary = pd.DataFrame({
    'Split':    ['Row-level (original)', 'Molecule-level (corrected)'],
    'RF ROC':   [orig_rf_roc,  rf_roc],
    'MLP ROC':  [orig_mlp_roc, mlp_roc],
    'RF PRC':   [0.8752,       rf_prc],
    'MLP PRC':  [0.889,        mlp_prc],
    'Note':     ['Data leakage present', 'Scientifically valid'],
})
print(summary.to_string(index=False))
print()
print(f"ROC-AUC drop from fixing leakage:")
print(f"  RF : {orig_rf_roc:.4f} → {rf_roc:.4f}  (Δ = {rf_roc-orig_rf_roc:+.4f})")
print(f"  MLP: {orig_mlp_roc:.4f} → {mlp_roc:.4f}  (Δ = {mlp_roc-orig_mlp_roc:+.4f})")
print()
print(f"Repurposing screen improvement:")
print(f"  Original : 3,112 candidates, top hits = known antibiotics")
print(f"  Fixed    : {len(df_hits):,} deduplicated non-antibiotic candidates")
print(f"  High-confidence (p>=0.8): {(df_hits.prob_ensemble>=0.8).sum()} true repurposing hits")